In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date
from utils.s3_aws import archive_landing

In [2]:
from pyspark.sql.functions import current_timestamp, lit

In [3]:
spark = get_sparkSession("Load bronze.fact_inventory")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp

In [6]:
_schema_name = "bronze"
_table_name = "fact_inventory"
_fullname = f"{_schema_name}.{_table_name}"
_filename = "fact_inventory.csv"
_file_path = f"s3a://ecommerce-pooja/pipeline/landing/fact_inventory/{_filename}"
_script_name = "raw_fact_inventory.ipynb"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for bronze.fact_inventory


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Reading csv file & creating dataframe

df = spark.read.format("csv") \
          .option("header", True) \
          .load(_file_path) \
          .dropDuplicates(["inventory_date", "product_id", "store", "channel"])


loaded_count = df.count()

In [9]:
df.show(10)
df.printSchema()
print(f"SPARK-APP : Rows entered : {loaded_count}")

+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+
|inventory_date|product_id| store|channel|currency|quantity_on_hand|quantity_reserved|quantity_available|cost_price|
+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+
|      20250101|     P1001|AMZ_US|    MKT|     USD|             100|               10|                90|       350|
|      20250101|     P1002|  EBAY|    MKT|     USD|             100|               10|                90|       350|
|      20250101|     P1003|  EBAY|    MKT|     USD|             100|               10|                90|       350|
|      20250102|     P1001|AMZ_US|    MKT|     USD|              98|               10|                88|       350|
|      20250102|     P1002|  EBAY|    MKT|     USD|             100|               10|                90|       350|
|      20250102|     P1003|  EBAY|    MKT|     USD|             

In [10]:

df_ld = df.withColumn("created_on", current_timestamp()) \
       .withColumn("created_by", lit(_script_name)) \
       .withColumn("modified_on", current_timestamp()) \
       .withColumn("modified_by", lit(_script_name)) \
       .withColumn("source_file", lit("fact_inventory.csv"))

In [11]:
df_ld.show(10)

+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+--------------------+--------------------+--------------------+--------------------+------------------+
|inventory_date|product_id| store|channel|currency|quantity_on_hand|quantity_reserved|quantity_available|cost_price|          created_on|          created_by|         modified_on|         modified_by|       source_file|
+--------------+----------+------+-------+--------+----------------+-----------------+------------------+----------+--------------------+--------------------+--------------------+--------------------+------------------+
|      20250101|     P1001|AMZ_US|    MKT|     USD|             100|               10|                90|       350|2026-01-29 00:29:...|raw_fact_inventor...|2026-01-29 00:29:...|raw_fact_inventor...|fact_inventory.csv|
|      20250101|     P1002|  EBAY|    MKT|     USD|             100|               10|                90|       350|2026

In [12]:
max_timestamp = get_max_loadTimestamp(spark, _schema_name, _table_name)

_data = []
if max_timestamp != '1900-01-01 00:00:00.000000':
    df_ld.write \
      .format("delta") \
      .mode("append") \
      .saveAsTable(_fullname)
    
    _data.append([_schema_name, _schema_name, _table_name, "delta-load", "append", str(datetime.now()), "success", loaded_count, _script_name, _script_name])
    insert_control_logs(spark, _data)
    
else:
    df_ld.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable(_fullname)
    
    _data.append([_schema_name, _schema_name, _table_name, "full-load", "overwrite", str(datetime.now()), "success", loaded_count, _script_name, _script_name])
    insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to {_schema_name}.{_table_name} and load_controller")

SPARK-APP: Data written to bronze.fact_inventory and load_controller


In [13]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+------+-----------+--------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| layer|schema_name|    table_name|load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+------+-----------+--------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|bronze|     bronze|fact_inventory|full-load| overwrite|2026-01-29 00:29:...|    success|           6|2026-01-29 00:29:...|raw_fact_inventor...|2026-01-29 00:29:...|raw_fact_inventor...|
+------+-----------+--------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [14]:
from delta import DeltaTable

dt = DeltaTable.forName(spark,f"{_schema_name}.{_table_name}")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|0      |6            |
+-------+-------------+



In [15]:
#Archive files

if archive_landing(_filename, "fact_inventory"):
    print(f"SPARK-APP: File {_filename} archived")
else:
    print(f"SPARK-APP: Error while archiving {_filename} from landing")

SPARK-APP: File fact_inventory.csv archived


In [16]:
spark.stop()